In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Mapping, Sequence
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import pulp

In [ ]:
# Constants

INPUT_BASE_PATH = Path("../data")
FORECASTS_PATH = Path("../results/ets/submissions")

# ==================================================
# OPTIMIZATION CONSTANTS / PARAMETERS / CONSTRAINTS
# ==================================================
TOTAL_STORAGE_CAPACITY = 35_000

# Gamma distributed
FOOD_VOLUME_DIST_PARAMS = {"shape": 5, "scale": 10}
HOUSEHOLD_VOLUME_DIST_PARAMS = {"shape": 10, "scale": 15}
HOBBIES_VOLUME_DIST_PARAMS = {"shape": 15, "scale": 17}

# Uniformly distributed
FOOD_STORAGE_COST_DIST_PARAMS = {"low": 0.1, "high": 0.5}
HOUSEHOLD_STORAGE_COST_DIST_PARAMS = {"low": 0.2, "high": 0.3}
HOBBIES_STORAGE_COST_DIST_PARAMS = {"low": 0.3, "high": 0.7}

FOODS_PRODUCT_IDS = [
    "FOODS_3_555_TX_2",
    "FOODS_3_376_TX_2",
    "FOODS_3_811_CA_2",
    "FOODS_1_218_TX_1",
    "FOODS_3_226_WI_3",
    "FOODS_3_070_WI_2",
    "FOODS_3_007_TX_2",
    "FOODS_3_444_TX_1",
    "FOODS_2_398_WI_3",
    "FOODS_3_540_WI_1",
]
HOUSEHOLD_PRODUCT_IDS = [
    "HOUSEHOLD_1_334_TX_1",
    "HOUSEHOLD_1_459_CA_2",
    "HOUSEHOLD_2_342_WI_2",
    "HOUSEHOLD_1_465_TX_3",
    "HOUSEHOLD_1_294_WI_2",
    "HOUSEHOLD_2_176_CA_3",
    "HOUSEHOLD_1_334_TX_2",
    "HOUSEHOLD_1_106_WI_2",
    "HOUSEHOLD_1_474_TX_2",
    "HOUSEHOLD_1_106_TX_1",
]
HOBBIES_PRODUCT_IDS = [
    "HOBBIES_1_048_WI_1",
    "HOBBIES_1_067_CA_3",
    "HOBBIES_1_158_TX_3",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_234_CA_3",
    "HOBBIES_1_254_CA_3",
    "HOBBIES_1_019_WI_1",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]
ALL_PRODUCT_IDS = list(FOODS_PRODUCT_IDS + HOUSEHOLD_PRODUCT_IDS + HOBBIES_PRODUCT_IDS)

# ==================
# FORECAST CONSTANTS 
# ==================
MAX_TRAINING_TIMESTAMP = 1913
FORECAST_HORIZON = 7
N_FORECAST_TIMESTAMPS = 4

# Timestamps at which we will generate forecasts. We will assume that we have already
# observed the ground truth value of each product series at the forecast timestamp and
# we produce forcasts for the following FORECAST_HORIZON timestamps.
FORECAST_TIMESTAMPS = [MAX_TRAINING_TIMESTAMP + i * FORECAST_HORIZON for i in range(N_FORECAST_TIMESTAMPS)]
FORECASTED_TIMESTAMPS = [list(range(t + 1, t + 1 + FORECAST_HORIZON)) for t in FORECAST_TIMESTAMPS]

RNG = np.random.default_rng(42)

#### RAW DATA

In [ ]:
# Load data

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")

In [ ]:
# Unpivot SALES_DF to long df

SALES_DF = (
    SALES_TRAIN_EVALUATION
    .with_columns(id=pl.col("id").str.strip_suffix("_evaluation"))
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales"
    ).with_columns(d_index=pl.col("d").str.extract(r"^d_([0-9]+)").cast(pl.Int64))
)

SALES_DF.head()

#### SELL PRICES

In [ ]:
PRICES_DF = (
    SALES_DF
    .select(pl.col("id", "item_id", "store_id", "d", "d_index"))
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .join(
        other=CALENDAR_DATA.select(pl.col("wm_yr_wk"), pl.col("d")),
        how="left",
        on="d",
        validate="m:1",
    )
    .join(
        other=SELL_PRICES,
        on=["item_id", "store_id", "wm_yr_wk"],
        how="left",
    )
)

PRICES_DF.head()

#### STORAGE PRICES

In [ ]:
# Define storage cost per item as a proportion of average sell
# price for item.

average_sell_prices = (
    SELL_PRICES
    .with_columns(
        id=pl.format("{}_{}", pl.col("item_id"), pl.col("store_id")),
        dept_id=pl.col("item_id").str.extract(r"^([^_]+)")
    )
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .group_by(pl.col("id"), pl.col("dept_id"))
    .agg(avg_sell_price=pl.col("sell_price").mean())
    .sort(pl.col("dept_id"), pl.col("id"))
)

food_storage_costs_pct = RNG.uniform(**FOOD_STORAGE_COST_DIST_PARAMS, size=len(FOODS_PRODUCT_IDS))
food_product_id_to_cost_pct = pl.DataFrame({"id": FOODS_PRODUCT_IDS, "cost_pct": food_storage_costs_pct})

household_storage_costs_pct = RNG.uniform(**HOUSEHOLD_STORAGE_COST_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_IDS))
household_product_id_to_cost_pct = pl.DataFrame({"id": HOUSEHOLD_PRODUCT_IDS, "cost_pct": household_storage_costs_pct})

hobbies_storage_costs_pct = RNG.uniform(**HOBBIES_STORAGE_COST_DIST_PARAMS, size=len(HOBBIES_PRODUCT_IDS))
hobbies_product_id_to_cost_pct = pl.DataFrame({"id": HOBBIES_PRODUCT_IDS, "cost_pct": hobbies_storage_costs_pct})

COST_DF = (
    average_sell_prices.join(
        other=food_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_food"}),
        how="left",
        on="id",
    ).join(
        other=household_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_household"}),
        how="left",
        on="id"
    ).join(
        other=hobbies_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_hobbies"}),
        how="left",
        on="id"
    ).with_columns(
        cost_pct=pl.coalesce(pl.selectors.starts_with("cost_pct_"))
    ).with_columns(
        storage_cost=(pl.col("cost_pct") * pl.col("avg_sell_price")).round(2)
    ).drop(
        pl.selectors.starts_with("cost_pct"),
        pl.col("avg_sell_price"),
        pl.col("dept_id"),
    )
)

COST_DF

#### STORAGE VOLUME

In [ ]:
# Define storage volume per product

food_product_volumes = RNG.gamma(**FOOD_VOLUME_DIST_PARAMS, size=len(FOODS_PRODUCT_IDS))
food_product_id_to_volume = pl.DataFrame({"id": FOODS_PRODUCT_IDS, "volume": food_product_volumes})

household_product_volumes = RNG.gamma(**HOUSEHOLD_VOLUME_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_IDS))
household_product_id_to_volume = pl.DataFrame({"id": HOUSEHOLD_PRODUCT_IDS, "volume": household_product_volumes})

hobbies_product_volumes = RNG.gamma(**HOBBIES_VOLUME_DIST_PARAMS, size=len(HOBBIES_PRODUCT_IDS))
hobbies_product_id_to_volume = pl.DataFrame({"id": HOBBIES_PRODUCT_IDS, "volume": hobbies_product_volumes})

VOLUME_DF = (
    SALES_DF
    .select(["id"])
    .unique()
    .join(
        other=food_product_id_to_volume.rename({"volume": "volume_foods"}),
        on="id",
        how="left"
    ).join(
        other=household_product_id_to_volume.rename({"volume": "volume_household"}),
        on="id",
        how="left",
    ).join(
        other=hobbies_product_id_to_volume.rename({"volume": "volume_hobbies"}),
        on="id",
        how="left"
    )
    .with_columns(
        storage_volume=pl.coalesce(pl.selectors.starts_with("volume_"))
    ).drop(
        pl.selectors.starts_with("volume_"),
    )
)

VOLUME_DF

#### SALES FORECASTS

In [ ]:
# Baseline forecasts

class M5HistoricalAverageModel:
    def __init__(self, lookback: int = 1, horizon: int = 1):
        self.lookback = lookback
        self.horizon = horizon

        self._product_ids: tuple[str, ...] | None = None
        self._max_train_index_by_product: dict[str, int] | None = None

    @property
    def product_ids(self) -> tuple[str, ...] | None:
        return self._product_ids

    def fit(self, train_df: pl.DataFrame) -> "M5HistoricalAverageModel":
        # Get unique product ids
        self._product_ids = tuple(train_df["id"].unique())

        # Calculate min/max timestamps for each product.
        start_end_index = (
            train_df
            .group_by(["id"])
            .agg(t_end=pl.col("d_index").max())
            .with_columns(t_start=pl.col("t_end") - self.lookback)
        )
        self._max_train_index_by_product = {
            d["id"]: d["t_end"] 
            for d in start_end_index[["id", "t_end"]].to_dicts()
        }

        # Join back onto train_df, filter to index within range
        # and calculate average
        avg_sales = (
            train_df
            .join(start_end_index, on="id", how="inner")
            .filter(pl.col("d_index").is_between(pl.col("t_start"), pl.col("t_end"), closed="both"))
            .group_by(["id"])
            .agg(avg_sales=pl.col("sales").mean())
        )
        self._avg_sales_by_product = {
            d["id"]: d["avg_sales"] 
            for d in avg_sales[["id", "avg_sales"]].to_dicts()
        }
        
        return self

    def predict(self, horizon: int | None = None) -> pl.DataFrame:
        if horizon is None:
            horizon = self.horizon
        elif isinstance(horizon, int) and horizon < 1:
            raise ValueError(f"Horizon has to be > 1. Got {horizon=}")
        
        forecast_dfs: list[pl.DataFrame] = []
        for product_id in self.product_ids:
            max_t_for_product = self._max_train_index_by_product[product_id]
            avg_sales_for_product = self._avg_sales_by_product[product_id]
            product_forecast_df = pl.DataFrame(
                {
                    "id": [product_id for _ in range(horizon)],
                    "F_index": [max_t_for_product + i for i in range(1, horizon + 1)],
                    "sales": [avg_sales_for_product for _ in range(horizon)]
                }
            )
            forecast_dfs.append(product_forecast_df)

        return pl.concat(forecast_dfs)



In [ ]:
# Generate forecasts at each forecast timestamp.
# The ground truth sales observations are observed at each forecast timestamp
# (i.e. they are included in the train data) and forecasts are generated
# for each timestamp in the forecast horizon after the forecast timestamp.

forecast_timestamps = [MAX_TRAINING_TIMESTAMP + i for i in range(2)]

for forecast_timestamp in forecast_timestamps:

    # Generate sales forecasts
    sales_train_df = SALES_DF.filter(
        pl.col("d_index") <= forecast_timestamp
    )
    sales_valid_df = SALES_DF.filter(
        pl.col("d_index") >= forecast_timestamp + 1,
        pl.col("d_index") <= forecast_timestamp + FORECAST_HORIZON,
    )
    
    model = M5HistoricalAverageModel(lookback=7, horizon=7)
    model.fit(sales_train_df)
    sales_forecast_df = model.predict()

    # Get sales prices
    sell_prices_df = (
        PRICES_DF.filter(
            pl.col("d_index") >= forecast_timestamp + 1,
            pl.col("d_index") <= forecast_timestamp + FORECAST_HORIZON,
        )
    )

    break

In [ ]:
product_id = SALES_DF["id"].sample(1).item()

fig, ax = plt.subplots()

sales_train_df = SALES_DF.filter(
    pl.col("id") == product_id,
    pl.col("d_index").is_between((MAX_TRAINING_TIMESTAMP - 7), MAX_TRAINING_TIMESTAMP)
).sort(by="d_index")

ax.plot(
    sales_train_df["d_index"].to_list(),
    sales_train_df["sales"].to_list(),
)
# ax.axhline(train_df["sales"].mean())

product_forecast_df = sales_forecast_df.filter(pl.col('id') == product_id).sort(by="F_index")
ax.plot(
    product_forecast_df["F_index"].to_list(),
    product_forecast_df["sales"].to_list(),
)

product_valid_df = sales_valid_df.filter(pl.col('id') == product_id).sort(by="d_index")
ax.plot(
    product_valid_df["d_index"].to_list(),
    product_valid_df["sales"].to_list(),
)

In [ ]:
@dataclass
class State:
    initial_stock: Mapping[str, int]


@dataclass
class ExogenousProblemData:
    demand_forecasts: Mapping[str, Sequence[float]]
    sell_prices: Mapping[str, Sequence[float]]
    storage_costs: Mapping[str, Sequence[float]]
    storage_volumes: Mapping[str, float]
    storage_capacity: float


@dataclass
class OptimalInventory:
    value_objective: float
    costs: Mapping[str, Sequence[float]]
    stock: Mapping[str, Sequence[int]]
    stock_additions: Mapping[str, Sequence[int]]
    sales: Mapping[str, Sequence[int]]
    

In [ ]:
def optimise_inventory(
    product_ids: Sequence[str],
    state: State,
    data: ExogenousProblemData,
    horizon: int = FORECAST_HORIZON,
    gamma: float = 0.95,
) -> pulp.LpProblem:
    # ==================
    # DEFINE LP PROBLEM

    # Description: At every timestamp we can add inventory to current stock and sell some inventory.
    # Any stock left over (i.e. not sold) at current timestamp is carried over to the next timestep.
    # Therefore stock at timestamp t is stock from previous timestep + stock addition at current 
    # timestep minus sales at current timestep.
    # ==================
    problem = pulp.LpProblem("mpc", sense=pulp.LpMinimize)

    timesteps = range(horizon)
    # ===============================
    # DECISION / AUXILIARY VARIABLES
    # ==============================
    stock_addition_var = pulp.LpVariable.dicts("stock_addition", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpInteger)
    stock_var = pulp.LpVariable.dicts("stock", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpInteger)

    cost_aux_var = pulp.LpVariable.dicts("cost_aux", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpContinuous)
    sales_aux_var = pulp.LpVariable.dicts("sales_aux", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpInteger)

    # ==================
    # Objective function

    # Minimise the cost auxiliary variable.
    # Cost auxiliary variable is a decomposition of costs from missing out on possible sales (under-stocking)
    # and cost from having to store more than what was sold (over-stocking). This objective is formulated via
    # constraints (see below).
    # ==================
    problem += pulp.lpSum(gamma ** k * cost_aux_var[p][k] for k in timesteps for p in product_ids)

    # ==================
    # Constraints
    # ==================

    # Constraint on sales to be min(demand, stock + addition).
    for k in timesteps:
        for p in product_ids:
            
            # Assume that we will not sell more than what was forecasted
            problem += sales_aux_var[p][k] <= data.demand_forecasts[p][k]
            
            # Cannot sell more than previous stock + stock addition at current timestamp
            prev_stock_level = state.initial_stock[p] if k == 0 else stock_var[p][k - 1]
            problem += sales_aux_var[p][k] <= prev_stock_level + stock_addition_var[p][k]


    for k in timesteps:
        for p in product_ids:

            # Cost for missing out on possible sales (stocking less than demand) - at optimum
            # cost from under-stocking have to be <= than minimum cost_aux_var
            problem += (
                data.sell_prices[p][k] * (data.demand_forecasts[p][k] - sales_aux_var[p][k])
            ) <= cost_aux_var[p][k]

            # Cost for storing more than what was sold - at optimum cost from over-stocking have
            # to be <= than minimum cost_aux_var
            prev_stock_level = state.initial_stock[p] if k == 0 else stock_var[p][k - 1]
            problem += (
                data.storage_costs[p][k]
                * (prev_stock_level + stock_addition_var[p][k] - sales_aux_var[p][k])
            ) <= cost_aux_var[p][k]


    # Storage capacity constraint
    for k in timesteps:
        storage_volume_all_products = []
        for p in product_ids:
            prev_stock_level = state.initial_stock[p] if k == 0 else stock_var[p][k - 1]
            product_storage_volume = (prev_stock_level + stock_addition_var[p][k]) *  data.storage_volumes[p]
            storage_volume_all_products.append(product_storage_volume)
        problem += pulp.lpSum(storage_volume_all_products) <= data.storage_capacity


    # Stock transition model
    for p in product_ids:
        for k in timesteps:
            prev_stock_level = state.initial_stock[p] if k == 0 else stock_var[p][k - 1]
            problem += stock_var[p][k] == (prev_stock_level + stock_addition_var[p][k] - sales_aux_var[p][k])

    
    # Solve problem
    problem.solve(pulp.PULP_CBC_CMD(msg=False))
    status = pulp.LpStatus[problem.status]
    if status != "Optimal":
        raise RuntimeError(f"Inventory optimisation failed with status {status!r}")

    
    # Extract solution
    costs = defaultdict(list)
    stock = defaultdict(list)
    stock_additions = defaultdict(list)
    sales = defaultdict(list)
    for p in product_ids:
        for k in timesteps:
            costs[p].append(cost_aux_var[p][k].value())
            stock[p].append(stock_var[p][k].value())
            stock_additions[p].append(stock_addition_var[p][k].value())
            sales[p].append(sales_aux_var[p][k].value())
    
    return OptimalInventory(
        value_objective=pulp.value(problem.objective),
        costs=dict(costs),
        stock=dict(stock),
        stock_additions=dict(stock_additions),
        sales=dict(sales),
    )


In [ ]:
# Solve for current forecast timestep

timesteps = range(FORECAST_HORIZON)
gamma = 0.95
product_ids = [
    "FOODS_3_376_TX_2",
    "FOODS_2_398_WI_3",
    "HOUSEHOLD_1_106_TX_1",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]

# Get forecasts by product id
product_demand_forecasts: dict[str, list[float]] = {}
for product_id in product_ids:
    product_forecast_df = sales_forecast_df.filter(pl.col("id") == product_id).sort(by='F_index')
    product_demand_forecasts[product_id] = product_forecast_df["sales"].to_list()


# Get sales prices by product id
product_sell_prices: dict[str, list[float]] = {}
for product_id in product_ids:
    product_sell_price_df = sell_prices_df.filter(pl.col("id") == product_id).sort(by="d_index")
    product_sell_prices[product_id] = product_sell_price_df["sell_price"].to_list()


# Get storage costs by product id
product_storage_costs: dict[str, list[float]] = {}
for product_id in product_ids:
    product_storage_cost = COST_DF.filter(pl.col("id") == product_id)["storage_cost"].item()
    product_storage_costs[product_id] = [product_storage_cost for _ in range(FORECAST_HORIZON)]


# Get storage volume by product id
product_storage_volumes: dict[str, float] = {}
for product_id in product_ids:
    product_storage_volume = VOLUME_DF.filter(pl.col("id") == product_id)["storage_volume"].item()
    product_storage_volumes[product_id] = product_storage_volume


# Initial stock levels
initial_product_stock: dict[str, int] = {}
for product_id in product_ids:
    initial_product_stock[product_id] = 2

# Construct state and data, and solve problem 
state = State(initial_stock=initial_product_stock)
data = ExogenousProblemData(
    demand_forecasts=product_demand_forecasts,
    sell_prices=product_sell_prices,
    storage_costs=product_storage_costs,
    storage_volumes=product_storage_volumes,
    storage_capacity=TOTAL_STORAGE_CAPACITY,
)
result = optimise_inventory(
    product_ids=product_ids,
    state=state,
    data=data,
    horizon=FORECAST_HORIZON,
    gamma=gamma,
)

In [ ]:
# Plot optimise_inventory() result

n_products = len(product_ids)

fig, axes = plt.subplots(n_products, 1, figsize=(10, n_products * 3.2), sharex=True, squeeze=False)
axes = axes.ravel()

for i, product_id in enumerate(product_ids):
    stock_var_values = np.concat([np.array([initial_product_stock[product_id]]), np.array(result.stock[product_id])])
    axes[i].plot(
        np.arange(FORECAST_HORIZON + 1),
        stock_var_values,
        color="steelblue",
        linewidth=1.5,
        marker="o",
        markersize=6,
        label="Stock",
    )

    axes[i].plot(
        np.arange(FORECAST_HORIZON) + 1,
        result.stock_additions[product_id],
        color="darkorange",
        marker="*",
        markersize=8,
        label="Stock addition",
    )

    axes[i].plot(
        np.arange(FORECAST_HORIZON) + 1,
        result.sales[product_id],
        color="seagreen",
        marker="^",
        ls="--",
        markersize=7,
        label="Sales",
    )

    axes[i].plot(
        np.arange(FORECAST_HORIZON) + 1,
        product_demand_forecasts[product_id],
        color="mediumpurple",
        marker="D",
        ls="--",
        markersize=5,
        label="Demand",
    )

    axes[i].set(ylabel="units", title=product_id, xticks=np.arange(FORECAST_HORIZON + 1))
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()
    axes[i].axvline(1, color="black", zorder=1, ls="--", alpha=0.75)

axes[-1].set_xlabel("Forecast day")
fig.tight_layout()
plt.show()
